# Recovery-time cohort plots

Converted from `plot_recovery.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Cohort-level recovery plots (all participants).

Outputs (default under Analysis_Results/cohort_recovery/):
  - plot3a_recovery_time_cohort.png — compact grouped bars (Recovery_s cohort mean ± SEM)
  - plot3b_trials_recovery_cohort.png — Trials_to_3CC: nine boxplots (Level 1–3 × Set 1–3)

Set 1 uses trial-level CSVs (BAPS_Subject*_trial_rt.csv); sets 2–3 use Trials_to_3CC from results.


In [ ]:
import argparse
import glob
import os
import re
import sys

os.environ.setdefault("MPLBACKEND", "Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 12})

COND_COLORS = {
    "Baseline": "#4C9BE8",
    "Distraction+Response": "#E84C4C",
    "Distraction+NoResponse": "#E8A94C",
}
# Display-only labels (the "NoResponse" condition key is glance-only/no-reply).
COND_SHORT_DISPLAY = {
    "Distraction+Response": "Response",
    "Distraction+NoResponse": "Glance",
}


def _load_all_results(data_dir: str) -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(data_dir, "BAPS_Subject*_results.csv")))
    if not paths:
        print(f"No BAPS_Subject*_results.csv in {data_dir}", file=sys.stderr)
        sys.exit(1)
    frames = []
    for p in paths:
        m = re.search(r"Subject(\d+)_results\.csv$", os.path.basename(p))
        if not m:
            continue
        sub = pd.read_csv(p)
        if "Participant" not in sub.columns:
            sub = sub.copy()
            sub["Participant"] = int(m.group(1))
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)


def _subject_mean_by_level_condition(
    df: pd.DataFrame, value_col: str
) -> pd.DataFrame:
    """One row per (Participant, Level, Condition): mean across tasks."""
    sub = df[df["Condition"] != "Baseline"].copy()
    if value_col not in sub.columns:
        return pd.DataFrame()
    sub = sub.dropna(subset=[value_col])
    if sub.empty:
        return pd.DataFrame()
    return (
        sub.groupby(["Participant", "Level", "Condition"], as_index=False)[value_col]
        .mean()
    )


def _cohort_mean_sem(agg: pd.DataFrame, value_col: str) -> pd.DataFrame:
    g = agg.groupby(["Level", "Condition"], as_index=False)[value_col].agg(
        mean="mean", sem=lambda s: s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else 0.0
    )
    return g


def _trials_to_first_3cc(correct: np.ndarray) -> float:
    streak = 0
    for i, v in enumerate(correct):
        if v == 1:
            streak += 1
            if streak >= 3:
                return float(i + 1)
        else:
            streak = 0
    return np.nan


def _split_set1_task_segments(trial_df: pd.DataFrame, level: int):
    sub = trial_df[(trial_df["Level"] == level) & (trial_df["Set"] == 1)].copy()
    if sub.empty:
        return []
    t = sub["Trial"].to_numpy()
    starts = [0]
    for i in range(1, len(t)):
        if t[i] < t[i - 1]:
            starts.append(i)
    starts.append(len(sub))
    return [sub.iloc[starts[k] : starts[k + 1]] for k in range(len(starts) - 1)]


def _collect_set1_trials_to_3cc(data_dir: str) -> pd.DataFrame:
    rows: list[dict] = []
    for path in sorted(glob.glob(os.path.join(data_dir, "BAPS_Subject*_trial_rt.csv"))):
        m = re.search(r"Subject(\d+)_trial_rt\.csv$", os.path.basename(path))
        if not m:
            continue
        pid = int(m.group(1))
        trial_df = pd.read_csv(path)
        if "Correct" not in trial_df.columns:
            continue
        for level in (1, 2, 3):
            for seg in _split_set1_task_segments(trial_df, level):
                c = seg["Correct"].to_numpy()
                n = _trials_to_first_3cc(c)
                rows.append(
                    {
                        "Participant": pid,
                        "Level": level,
                        "Set1_trials_to_3cc": n,
                    }
                )
    return pd.DataFrame(rows)


def plot3a_recovery_time_cohort(df: pd.DataFrame, out_path: str) -> None:
    agg = _subject_mean_by_level_condition(df, "Recovery_s")
    if agg.empty:
        print("   Skipping 3a: no Recovery_s rows")
        return
    cohort = _cohort_mean_sem(agg, "Recovery_s")
    levels = [1, 2, 3]
    n = len(levels)
    x = np.arange(n, dtype=float)
    # Narrow grouped bars (“skyscraper” layout)
    width = 0.32
    gap = 0.02
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    conds = ["Distraction+Response", "Distraction+NoResponse"]
    offsets = [-width / 2 - gap / 2, width / 2 + gap / 2]
    for cond, dx in zip(conds, offsets):
        sub = cohort[cohort["Condition"] == cond].set_index("Level").reindex(levels)
        y = sub["mean"].values.astype(float)
        e = sub["sem"].values.astype(float)
        color = COND_COLORS.get(cond, "#333333")
        pos = x + dx
        ax.bar(
            pos,
            y,
            width,
            yerr=e,
            capsize=2,
            error_kw={"linewidth": 0.9, "ecolor": "#333"},
            label=COND_SHORT_DISPLAY.get(cond, cond),
            color=color,
            edgecolor="white",
            linewidth=0.6,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f"L{l}" for l in levels], fontsize=13, fontweight="bold")
    ax.set_xlabel("Level", fontsize=14, fontweight="bold")
    ax.set_ylabel("Recovery time (s)", fontsize=14, fontweight="bold")
    ax.tick_params(axis="y", labelsize=12)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.legend(fontsize=12, framealpha=0.9)
    fig.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Saved: {out_path}")


def plot3b_trials_cohort(df: pd.DataFrame, set1_df: pd.DataFrame, out_path: str) -> None:
    """Nine boxplots: Level 1–3 × Set 1–3 (Set 1 from trial_rt; sets 2–3 from results CSV)."""
    if "Trials_to_3CC" not in df.columns:
        print("   Skipping 3b: no Trials_to_3CC column")
        return

    sub23 = df[df["Set"].isin([2, 3])].dropna(subset=["Trials_to_3CC"])
    set1_df = set1_df if set1_df is not None else pd.DataFrame()
    if not set1_df.empty:
        set1_df = set1_df.dropna(subset=["Set1_trials_to_3cc"])

    levels = [1, 2, 3]
    groups: list[np.ndarray] = []
    box_colors: list[str] = []
    xticklabels: list[str] = []
    set_color = {
        1: COND_COLORS["Baseline"],
        2: COND_COLORS["Distraction+Response"],
        3: COND_COLORS["Distraction+NoResponse"],
    }

    for lv in levels:
        for set_id in (1, 2, 3):
            if set_id == 1:
                if set1_df.empty:
                    vals = np.array([], dtype=float)
                else:
                    vals = set1_df[set1_df["Level"] == lv]["Set1_trials_to_3cc"].to_numpy(dtype=float)
                    vals = vals[~np.isnan(vals)]
            else:
                vals = sub23[(sub23["Level"] == lv) & (sub23["Set"] == set_id)][
                    "Trials_to_3CC"
                ].to_numpy(dtype=float)
            groups.append(vals)
            box_colors.append(set_color[set_id])
            xticklabels.append(f"L{lv}\nS{set_id}")

    if all(len(g) == 0 for g in groups):
        print("   Skipping 3b: no data for any level×set")
        return

    positions = np.arange(len(groups), dtype=float)
    box_w = 0.48
    fig, ax = plt.subplots(figsize=(14, 5.4))

    flier_kw = dict(
        marker="o",
        markersize=8,
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=1.4,
        linestyle="none",
    )
    bp = ax.boxplot(
        groups,
        positions=positions,
        widths=box_w,
        patch_artist=True,
        showfliers=True,
        flierprops=flier_kw,
    )
    for patch, c in zip(bp["boxes"], box_colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.55)
        patch.set_edgecolor("#222222")
        patch.set_linewidth(1.1)
    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(2.2)
    for w in bp["whiskers"]:
        w.set_color("#333333")
        w.set_linewidth(1.0)
    for cap in bp["caps"]:
        cap.set_color("#333333")
        cap.set_linewidth(1.0)

    text_x_off = box_w / 2 + 0.12
    for pos, g in zip(positions, groups):
        if len(g) == 0:
            continue
        mn = float(np.mean(g))
        # Mean: diamond (median is the horizontal segment inside each box from boxplot)
        ax.plot(
            pos,
            mn,
            marker="D",
            markersize=10,
            markerfacecolor="black",
            markeredgecolor="black",
            linestyle="none",
            zorder=6,
            clip_on=False,
        )
        ax.text(
            pos + text_x_off,
            mn,
            f"{mn:.2f}",
            ha="left",
            va="center",
            fontsize=11,
            fontweight="bold",
            color="black",
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(xticklabels, fontsize=12, fontweight="bold")
    ax.set_xlabel("Level × Set", fontsize=14, fontweight="bold")
    ax.set_ylabel("Trials to 3 consecutive correct", fontsize=14, fontweight="bold")
    ax.tick_params(axis="y", labelsize=12)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.set_xlim(positions[0] - 0.55, positions[-1] + text_x_off + 0.72)

    ax.legend(
        handles=[
            Line2D(
                [0],
                [0],
                marker="D",
                color="none",
                markerfacecolor="black",
                markeredgecolor="black",
                markersize=10,
                linestyle="none",
                label="Mean",
            ),
            Line2D([0], [0], color="black", linewidth=2.5, linestyle="-", label="Median"),
        ],
        loc="upper left",
        framealpha=0.92,
        fontsize=12,
    )

    fig.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   Saved: {out_path}")


In [ ]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument(
        "--data-dir",
        default=str(_ANALYSIS_DIR),
        help="Folder containing BAPS_Subject*_results.csv and BAPS_Subject*_trial_rt.csv (for Set 1)",
    )
    ap.add_argument(
        "--out-dir",
        default=None,
        help="Output directory (default: <data-dir>/Analysis_Results/cohort_recovery)",
    )
    args = ap.parse_args()
    data_dir = os.path.abspath(args.data_dir)
    out_dir = args.out_dir or os.path.join(
        data_dir, "Analysis_Results", "cohort_recovery"
    )
    os.makedirs(out_dir, exist_ok=True)

    print(f"Data dir: {data_dir}")
    print(f"Out dir:  {out_dir}")
    df = _load_all_results(data_dir)
    print(f"Loaded {df['Participant'].nunique()} participants, {len(df)} rows")

    set1_df = _collect_set1_trials_to_3cc(data_dir)
    if not set1_df.empty:
        print(f"Set 1 trials-to-3CC (trial_rt): {len(set1_df)} participant×task values")

    plot3a_recovery_time_cohort(df, os.path.join(out_dir, "plot3a_recovery_time_cohort.png"))
    plot3b_trials_cohort(
        df,
        set1_df,
        os.path.join(out_dir, "plot3b_trials_recovery_cohort.png"),
    )


In [ ]:
# Run this cell to execute the script
main()
